## Importation des packages

In [ ]:

# Importer les modules nécessaires
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

## Chargement des données

Les données concernent  des informations sur les paiements en défaut, les facteurs démographiques, les données de crédit, l'historique des paiements et les relevés de compte des clients de cartes de crédit à Taïwan d'avril 2005 à septembre 2005.

In [ ]:
whine_quality = pd.read_csv('WineQT.csv', sep=',')

In [ ]:

whine_quality.head()

## Informations sur l'ensemble de données

Cet ensemble de données contient des informations sur les paiements en défaut, les facteurs démographiques, les données de crédit, l'historique des paiements et les relevés de compte des clients de cartes de crédit à Taïwan d'avril 2005 à septembre 2005.

- ID : Identifiant de chaque client

- LIMIT_BAL : Montant du crédit accordé en dollars NT (incluant les crédits individuels et familiaux/supplémentaires)

- SEXE : Genre (1=homme, 2=femme)

- ÉDUCATION : (1=études supérieures, 2=université, 3=lycée, 4=autres, 5=inconnu, 6=inconnu)

- MARRIAGE : État civil (1=marié, 2=célibataire, 3=autre)

- ÂGE : Âge en années

- PAY_0 : État du remboursement en septembre 2005 (-1=paiement à l'échéance, 1=retard de paiement d'un mois, 2=retard de paiement de deux mois, … 8=retard de paiement de huit mois, 9=retard de paiement de neuf mois et plus)

- PAY_2 : État des remboursements en août 2005 (échelle identique à celle ci-dessus)

- PAY_3 : État du remboursement en juillet 2005 (échelle identique à celle ci-dessus)

- PAY_4 : État des remboursements en juin 2005 (échelle identique à celle ci-dessus)

- PAY_5 : État du remboursement en mai 2005 (échelle identique à celle ci-dessus)

- PAY_6 : État des remboursements en avril 2005 (échelle identique à celle ci-dessus)

- BILL_AMT1 : Montant de la facture de septembre 2005 (dollar NT)

- BILL_AMT2 : Montant de la facture en août 2005 (dollar NT)

- BILL_AMT3 : Montant de la facture de juillet 2005 (dollar NT)

- BILL_AMT4 : Montant de la facture de juin 2005 (dollar NT)

- BILL_AMT5 : Montant de la facture de mai 2005 (dollar NT)

- BILL_AMT6 : Montant de la facture d'avril 2005 (dollar NT)

- PAY_AMT1 : Montant du paiement précédent en septembre 2005 (dollar NT)

- PAY_AMT2 : Montant du paiement précédent en août 2005 (dollar NT)

- PAY_AMT3 : Montant du paiement précédent en juillet 2005 (dollar NT)

- PAY_AMT4 : Montant du paiement précédent en juin 2005 (dollar NT)

- PAY_AMT5 : Montant du paiement précédent en mai 2005 (dollar NT)

- PAY_AMT6 : Montant du paiement précédent en avril 2005 (dollar NT)

- default.payment.next.month : Paiement par défaut (1=oui, 0=non)

## Exploration des données 

In [ ]:
whine_quality.info()

In [ ]:
whine_quality.describe(include='all')


In [ ]:
whine_quality.isnull().sum()

In [ ]:
# Distribution de la variable cible 'quality'
plt.figure(figsize=(8, 6))
whine_quality['quality'].value_counts().plot(kind='bar')
plt.title('Distribution de la qualité du vin')
plt.xlabel('Qualité')
plt.ylabel('Nombre d\'observations')
plt.show()

In [ ]:
# Créer une nouvelle version de la base de données sans les variables "quality" et "id"
whine_quality_new = whine_quality.drop(columns=['quality', 'Id'], axis=1)
whine_quality_new.head()

In [ ]:
# Générer un box plot pour chaque variable 
for var in whine_quality_new.columns:
    whine_quality_new[var].plot(kind='box')
    plt.title(var)
    plt.show()

## Analyse bivariée et création des classes

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Créer trois classes pour la variable quality: Bad (3,4), Average (5,6) et Good (7,8)
whine_quality['quality'] = whine_quality['quality'].apply(
    lambda x: 'Bad' if (x <= 4) else ('Average' if (x <= 6) else 'Good')
)

# Définir l'ordre des catégories pour un affichage logique
quality_order = ['Bad', 'Average', 'Good']

# Générer un box plot pour chaque variable en fonction de la classe de qualité
for var in whine_quality.columns:
    if var != 'quality': 
        plt.figure(figsize=(8, 5))  
        sns.boxplot(x='quality', y=var, data=whine_quality, order=quality_order)
        plt.title(f'Distribution de {var} par classe de qualité')
        plt.xlabel('Classe de qualité')
        plt.ylabel(var)
        plt.show()

In [ ]:
# Créer une colonne supplémentaire pour la classe de qualité ( Bad=0, Average=1, Good=2)
quality_mapping = {'Bad': 0, 'Average': 1, 'Good': 2}
whine_quality_new['quality_class'] = whine_quality['quality'].map(quality_mapping)

whine_quality_new.head()

In [ ]:
import scipy.stats as stats
# Initialiser les listes pour stocker les résultats
var_names = []
kw_stats = []
p_values = []

# Parcourir toutes les variables numériques
for var in whine_quality_new.columns:
    # Calculer les groupes de valeurs
    groups = [whine_quality_new[whine_quality_new['quality_class'] == 0][var], whine_quality_new[whine_quality_new['quality_class'] == 1][var], whine_quality_new[whine_quality_new['quality_class'] == 2][var]]
    # Appliquer le test de Kruskal-Wallis
    kw_stat, p = stats.kruskal(*groups)
    # Ajouter les résultats aux listes correspondantes
    var_names.append(var)
    kw_stats.append(kw_stat)
    p_values.append(p)

# Créer un DataFrame avec les résultats
results_df = pd.DataFrame({
    'Variable': var_names,
    'Kruskal-Wallis': kw_stats,
    'P-valeur': p_values
})

# Trier le DataFrame par ordre croissant de p-valeur
results_df.sort_values(by='P-valeur', inplace=True)

# Afficher le tableau des résultats
print(results_df)

## Modélisation

In [ ]:
import statsmodels.api as sm
# Sélectionner les variables explicatives et la variable d'intérêt
X = whine_quality_new[['fixed acidity', 'volatile acidity', 'citric acid',  'chlorides',  'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]
y = whine_quality_new['quality_class']

In [ ]:
from sklearn.preprocessing import StandardScaler
# Ajouter une constante pour l'interception
X = sm.add_constant(X)

# Diviser les données en ensembles d'apprentissage et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


In [ ]:

# Standardiser les données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Créer le modèle de régression logistique
model = LogisticRegression(
    solver='lbfgs',
    max_iter=1250,
    class_weight=None,
    random_state=42
)

In [ ]:
# Ajuster le modèle aux données d'apprentissage
result = model.fit(X_train_scaled, y_train)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Prédictions pour Train
y_train_pred = result.predict(X_train_scaled)

# Prédictions pour Test
y_test_pred = result.predict(X_test_scaled)


print("CLASSIFICATION REPORT - ENSEMBLE D'ENTRAÎNEMENT")
print(classification_report(y_train, y_train_pred, 
                          target_names=['Bad', 'Average', 'Good']))

print("CLASSIFICATION REPORT - ENSEMBLE DE TEST \n")

print(classification_report(y_test, y_test_pred, 
                          target_names=['Bad', 'Average', 'Good']))



## Oversampling

In [ ]:


from imblearn.over_sampling import RandomOverSampler

sampling_strategy = {
    0: 703,
    1: 703,
    2: 703
}

ros = RandomOverSampler(
    sampling_strategy=sampling_strategy,
    random_state=42
)

X_train_os, y_train_os = ros.fit_resample(X_train_scaled, y_train)


X_train_oversampled, y_train_oversampled = ros.fit_resample(X_train_scaled, y_train)

# Créer un nouveau DataFrame avec les données oversampled
bankdata_oversampled = pd.concat([X_train_oversampled, y_train_oversampled], axis=1)

In [ ]:
# Fonction pour créer un pie chart avec les proportions et les nombres
def plot_pie_chart(y, title):
    labels = ['0', '1']
    sizes = y.value_counts().values
    colors = ['#66b3ff', '#ff9999']
    
    # Modifier le format des labels pour inclure les proportions et les nombres
    def autopct_format(pct, all_values):
        absolute = int(round(pct / 100 * sum(all_values)))
        return f"{pct:.1f}% ({absolute})"

    fig, ax = plt.subplots()
    ax.pie(sizes, labels=labels, colors=colors, autopct=lambda pct: autopct_format(pct, sizes), startangle=90)
    ax.axis('equal')  # Pour assurer que le diagramme est bien rond
    plt.title(title)
    plt.show()

# Créer un pie chart pour la table basique
plot_pie_chart(y_train, "Table basique - Distribution de 'default.payment.next.month'")

# Créer un pie chart pour la table oversampled
plot_pie_chart(y_train_oversampled, "Table oversampled - Distribution de 'default.payment.next.month'")

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

# Entraîner et évaluer la régression logistique sur la base basique
logreg_basique = sm.Logit(y_train, X_train).fit()
y_pred_basique = logreg_basique.predict(X_test)
y_pred_basique_train = logreg_basique.predict(X_train)
auc_basique = roc_auc_score(y_test, y_pred_basique)
auc_train_basique = roc_auc_score(y_train, y_pred_basique_train)
# Entraîner et évaluer la régression logistique sur la base oversampled
logreg_oversampled = sm.Logit(y_train_oversampled, X_train_oversampled).fit(disp=0)
y_pred_oversampled = logreg_oversampled.predict(X_test)
auc_oversampled = roc_auc_score(y_test, y_pred_oversampled)
y_pred_oversampled_train = logreg_oversampled.predict(X_train_oversampled)
auc_oversampled_train = roc_auc_score(y_train_oversampled, y_pred_oversampled_train)

In [ ]:
# Créer un DataFrame avec les performances
performances = pd.DataFrame({
    'Modèle': ['Base basique', 'Base oversampled'],
    'AUC - Entraînement': [auc_train_basique, auc_oversampled_train],
    'AUC - Test': [auc_basique, auc_oversampled]
})

# Afficher les performances
print(performances)

## Il n'y a pas eu de différence significative avec le modèle de base. 

Passons à l'undersampling

In [ ]:
# Importer les bibliothèques nécessaires
from imblearn.under_sampling import RandomUnderSampler

# Initialiser l'objet RandomUnderSampler
rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)

# Appliquer l'undersampling sur les données d'apprentissage
X_train_undersampled, y_train_undersampled = rus.fit_resample(X_train, y_train)

# Créer un nouveau DataFrame avec les données undersampled
credit_data_undersampled = pd.concat([X_train_undersampled, y_train_undersampled], axis=1)

# Afficher la nouvelle distribution des données
print(credit_data_undersampled['default.payment.next.month'].value_counts())

In [ ]:
# Créer un pie chart pour la table undersampled
plot_pie_chart(y_train_undersampled, "Table undersampled - Distribution de 'default.payment.next.month'")

In [ ]:
# Entraîner et évaluer la régression logistique sur la base oversampled
lr_undersampled = sm.Logit(y_train_undersampled, X_train_undersampled).fit()


y_pred_undersampled = lr_undersampled.predict(X_test)
auc_undersampled = roc_auc_score(y_test, y_pred_undersampled)

# Ajouter les performances de la base undersampled au DataFrame
performances.loc[2] = ['Base undersampled', auc_undersampled, auc_undersampled]

# Afficher les performances
print(performances)